# Regression of Vote Characteristics on CAR

In [1]:
import stata_setup

stata_setup.config('/Applications/StataNow', 'mp')


  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      StataNow 19.5
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2025 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Unlimited-user 2-core network, expiring  9 Sep 2026
Serial number: 501909305069
  Licensed to: 罗奕辰
               UCL

Notes:
      1. Unicode is supported; see help unicode_advice.
      2. More than 2 billion observations are allowed; see help obs_advice.
      3. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


In [2]:
%%stata

clear all
set more off
set varabbrev off
version 18

* Paths
global PROCESSED_DATA "processed_data"
global TABLES "tables"


. 
. clear all

. set more off

. set varabbrev off

. version 18

. 
. * Paths
. global PROCESSED_DATA "processed_data"

. global TABLES "tables"

. 


In [ ]:
%%stata

eststo clear

foreach stage in created end {
    import delimited using "$PROCESSED_DATA/event_study_panel_`stage'.csv", clear

    * Parse date
    gen day = date(date, "YMD")
    format day %td
    gen year = year(day)

    * Ensure index is numeric (safeguard if it came in as string)
    capture confirm numeric variable index
    if _rc {
        destring index, replace ignore("[]")
    }

    * Encode token for FE/clustering
    encode gecko_id, gen(token)

    * Label variable
    label var non_whale_victory_vn "\${\it Small Win}_{i,t}\$"

    * Keep CAR[-5,5] window
    keep if index == 5

    * Run regression
    reghdfe car non_whale_victory_vn, absorb(year token) vce(cluster token)
    eststo car_`stage'
    estadd local fe_token "Y"
    estadd local fe_time "Y"
}

* Export combined LaTeX table
esttab                                                           ///
    car_created car_end                                          ///
    using "$TABLES/reg_car_win.tex", replace                     ///
    se star(* 0.10 ** 0.05 *** 0.01) label nogaps nocon nonumbers ///
    nonotes booktabs                                             ///
    b(%9.3f) se(%9.2f)                                           ///
    mtitles("\${\it CAR^{\it Create}}_i\$" "\${\it CAR^{\it End}}_i\$") ///
    posthead(\cmidrule(lr){2-3} & \multicolumn{1}{c}{(1)} & \multicolumn{1}{c}{(2)} \\ \midrule) ///
    substitute("\_" "_")                                         ///
    stats(fe_token fe_time N r2_a,                               ///
        fmt(0 0 %9.0fc %9.3f)                                    ///
         labels("Token FE" "Year FE" "Observations" "Adjusted R²"))